In [4]:
"""
채용공고 지역 편중 지도 시각화
- 지도 1: 시·도별 단계구분도 (데이터분석가 / 백엔드개발자 각 1장)
- 지도 2: 서울 구별 단계구분도 (데이터분석가 / 백엔드개발자 각 1장)

※ 준비물 (통계청 SGIS 다운로드):
   - 시도 경계: sido.shp  (또는 GeoJSON)
   - 서울 구 경계: seoul_gu.shp  (또는 GeoJSON)
   SGIS: https://sgis.kostat.go.kr/view/map/preBoundaryLayer

※ 추천 대안 (바로 사용 가능):
   pip install git+https://github.com/southkorea/southkorea-maps  # 또는 아래 URL
   sido:     https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2018/json/skorea-provinces-2018-geo.json
   sigungu:  https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2018/json/skorea-municipalities-2018-geo.json
"""

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.ticker as mticker
import numpy as np
import os

# ────────────────────────────────────────────────
# 0. 경로 설정  ← 실제 파일 위치에 맞게 수정
# ────────────────────────────────────────────────
EXCEL_PATH   = r"C:\py_temp\중간프로젝트\posting_analysis_table_최종.xlsx"   # 분석 데이터
SIDO_SHP     = r"C:\py_temp\중간프로젝트\sido.geojson"                  # 시도 경계 파일
SEOUL_GU_SHP = r"C:\py_temp\중간프로젝트\seoul_gu.geojson"                 # 서울 구 경계 파일
OUTPUT_DIR   = r"C:\py_temp\중간프로젝트\output_maps_m1"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ────────────────────────────────────────────────
# 1. 데이터 로드 & 전처리
# ────────────────────────────────────────────────
df = pd.read_excel(EXCEL_PATH)

# 해외 지역 제거
FOREIGN_KEYWORDS = ['미국', '일본', '베트남', '헝가리', '동경', '일본 도쿄']
mask_foreign = df['시각화용_지역'].apply(
    lambda x: any(k in str(x) for k in FOREIGN_KEYWORDS)
)
df = df[~mask_foreign].copy()

print(f"국내 데이터: {len(df)}건")
print(df['job'].value_counts())

# ────────────────────────────────────────────────
# 2. 집계 함수
# ────────────────────────────────────────────────
def aggregate_sido(df, job_name):
    """시도별 집계"""
    sub = df[df['job'] == job_name].copy()
    total = len(sub)
    agg = sub.groupby('region_sido').size().reset_index(name='count')
    agg['pct'] = (agg['count'] / total * 100).round(1)
    return agg, total

def aggregate_seoul_gu(df, job_name):
    """서울 구별 집계 - 인천 중구 vs 서울 중구 구분"""
    sub = df[(df['job'] == job_name) & (df['region_sido'] == '서울')].copy()
    total_seoul = len(sub)
    
    # '시각화용_지역'에서 구 이름 추출 (앞에 '서울 ' prefix 제거)
    sub['gu'] = sub['시각화용_지역'].str.replace('^서울 ', '', regex=True).str.strip()
    sub.loc[sub['gu'] == '서울', 'gu'] = '기타'  # 구 정보 없는 경우
    
    agg = sub.groupby('gu').size().reset_index(name='count')
    agg['pct'] = (agg['count'] / total_seoul * 100).round(1)
    return agg, total_seoul


# ────────────────────────────────────────────────
# 3. 색상 팔레트 정의
# ────────────────────────────────────────────────
# 데이터분석가: 파란 계열
CMAP_DA = LinearSegmentedColormap.from_list(
    'da_blue',
    ['#EBF5FB', '#AED6F1', '#5DADE2', '#2874A6', '#1A5276'],
    N=256
)
# 백엔드개발자: 초록 계열
CMAP_BE = LinearSegmentedColormap.from_list(
    'be_green',
    ['#E9F7EF', '#A9DFBF', '#52BE80', '#1E8449', '#145A32'],
    N=256
)

JOB_CONFIG = {
    '데이터 분석가': {
        'cmap': CMAP_DA,
        'accent': '#2874A6',
        'label_color': 'white',
        'filename_prefix': 'DA',
    },
    '백엔드 개발자': {
        'cmap': CMAP_BE,
        'accent': '#1E8449',
        'label_color': 'white',
        'filename_prefix': 'BE',
    }
}


# ────────────────────────────────────────────────
# 4. 공통 스타일 설정
# ────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': ['Malgun Gothic', 'AppleGothic', 'NanumGothic',
                    'DejaVu Sans'],   # 한글 폰트 자동 선택
    'axes.unicode_minus': False,
})

# 한글 폰트 강제 설정 (Linux 환경)
import matplotlib.font_manager as fm
def set_korean_font():
    """시스템에서 한글 폰트를 찾아 설정"""
    candidates = [
        '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
        '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
        '/System/Library/Fonts/AppleGothic.ttf',  # macOS
        'C:/Windows/Fonts/malgun.ttf',             # Windows
    ]
    for path in candidates:
        if os.path.exists(path):
            fe = fm.FontEntry(fname=path, name='KoreanFont')
            fm.fontManager.ttflist.insert(0, fe)
            plt.rcParams['font.family'] = 'KoreanFont'
            print(f"폰트 설정: {path}")
            return True
    print("⚠ 한글 폰트를 찾지 못했습니다. 한글 깨짐이 발생할 수 있습니다.")
    return False

set_korean_font()


# ────────────────────────────────────────────────
# 5. 시도별 지도 그리기
# ────────────────────────────────────────────────
# 시도 이름 매핑 (GeoJSON 속성명 → region_sido)
# southkorea-maps 기준 'name' 컬럼 사용
SIDO_NAME_MAP = {
    '서울특별시': '서울', '부산광역시': '부산', '대구광역시': '대구',
    '인천광역시': '인천', '광주광역시': '광주', '대전광역시': '대전',
    '울산광역시': '울산', '세종특별자치시': '세종',
    '경기도': '경기', '강원특별자치도': '강원', '강원도': '강원',
    '충청북도': '충북', '충청남도': '충남',
    '전라북도': '전북', '전북특별자치도': '전북',
    '전라남도': '전남', '경상북도': '경북', '경상남도': '경남',
    '제주특별자치도': '제주',
    # 영문 키 (GeoJSON 파일에 따라 다를 수 있음)
    'Seoul': '서울', 'Busan': '부산', 'Daegu': '대구',
    'Incheon': '인천', 'Gwangju': '광주', 'Daejeon': '대전',
    'Ulsan': '울산', 'Sejong': '세종',
    'Gyeonggi-do': '경기', 'Gangwon-do': '강원',
    'Chungcheongbuk-do': '충북', 'Chungcheongnam-do': '충남',
    'Jeollabuk-do': '전북', 'Jeollanam-do': '전남',
    'Gyeongsangbuk-do': '경북', 'Gyeongsangnam-do': '경남',
    'Jeju-do': '제주',
}

def draw_sido_map(job_name, sido_gdf_raw, df, output_dir):
    """시도별 단계구분도 생성"""
    cfg = JOB_CONFIG[job_name]
    agg, total = aggregate_sido(df, job_name)
    
    # GeoJSON 이름 컬럼 찾기 (name 또는 CTP_KOR_NM 등)
    name_col = None
    for col in ['name', 'CTP_KOR_NM', 'SIDO_NM', 'kor_name', 'NAME_1']:
        if col in sido_gdf_raw.columns:
            name_col = col
            break
    if name_col is None:
        print("⚠ 시도 이름 컬럼을 찾지 못했습니다. 컬럼 목록:", sido_gdf_raw.columns.tolist())
        return

    sido_gdf = sido_gdf_raw.copy()
    # 이름 표준화
    sido_gdf['sido_std'] = sido_gdf[name_col].map(SIDO_NAME_MAP).fillna(sido_gdf[name_col])
    
    # 데이터 병합
    sido_gdf = sido_gdf.merge(agg, left_on='sido_std', right_on='region_sido', how='left')
    sido_gdf['count'] = sido_gdf['count'].fillna(0)
    sido_gdf['pct']   = sido_gdf['pct'].fillna(0)
    
    # ── 그리기 ──
    fig, ax = plt.subplots(1, 1, figsize=(12, 14), facecolor='#FAFAFA')
    ax.set_facecolor('#FAFAFA')
    
    vmax = sido_gdf['count'].max()
    vmin = 0
    
    sido_gdf.plot(
        column='count',
        cmap=cfg['cmap'],
        linewidth=0.6,
        edgecolor='#AAAAAA',
        ax=ax,
        vmin=vmin,
        vmax=vmax,
        missing_kwds={'color': '#EEEEEE'},
    )
    
    # 레이블: 건수 + 퍼센트
    for _, row in sido_gdf.iterrows():
        if row['count'] == 0:
            continue
        centroid = row.geometry.centroid
        x, y = centroid.x, centroid.y
        
        # 배경 밝기에 따라 글자색 결정
        norm_val = row['count'] / vmax if vmax > 0 else 0
        text_color = 'white' if norm_val > 0.45 else '#1A1A1A'
        
        sido_short = row['sido_std']
        label = f"{sido_short}\n{int(row['count'])}건\n({row['pct']}%)"
        
        ax.annotate(
            label,
            xy=(x, y),
            ha='center', va='center',
            fontsize=7.5,
            color=text_color,
            fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.15', fc='none', ec='none'),
        )
    
    # 컬러바
    sm = plt.cm.ScalarMappable(
        cmap=cfg['cmap'],
        norm=mcolors.Normalize(vmin=vmin, vmax=vmax)
    )
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02, shrink=0.6)
    cbar.set_label('공고 건수', fontsize=11)
    cbar.ax.tick_params(labelsize=9)
    
    # 제목
    title_map = {'데이터 분석가': '데이터 분석가', '백엔드 개발자': '백엔드 개발자'}
    ax.set_title(
        f'[시도별] {title_map[job_name]} 채용공고 분포\n(총 {total}건)',
        fontsize=16, fontweight='bold', pad=18, color='#1A1A1A'
    )
    ax.axis('off')
    
    plt.tight_layout()
    fname = f"{cfg['filename_prefix']}_sido_map.png"
    fpath = os.path.join(output_dir, fname)
    plt.savefig(fpath, dpi=200, bbox_inches='tight', facecolor='#FAFAFA')
    plt.close()
    print(f"저장: {fpath}")


# ────────────────────────────────────────────────
# 6. 서울 구별 지도 그리기
# ────────────────────────────────────────────────
# 서울 구 이름 매핑 (GeoJSON 속성 → 구 이름)
GU_NAME_MAP = {
    # 한글 전체명 → 약칭
    '강남구': '강남구', '강동구': '강동구', '강북구': '강북구', '강서구': '강서구',
    '관악구': '관악구', '광진구': '광진구', '구로구': '구로구', '금천구': '금천구',
    '노원구': '노원구', '도봉구': '도봉구', '동대문구': '동대문구', '동작구': '동작구',
    '마포구': '마포구', '서대문구': '서대문구', '서초구': '서초구', '성동구': '성동구',
    '성북구': '성북구', '송파구': '송파구', '양천구': '양천구', '영등포구': '영등포구',
    '용산구': '용산구', '은평구': '은평구', '종로구': '종로구', '중구': '중구',
    '중랑구': '중랑구',
}

def draw_seoul_map(job_name, seoul_gdf_raw, df, output_dir):
    """서울 구별 단계구분도 생성"""
    cfg = JOB_CONFIG[job_name]
    agg, total_seoul = aggregate_seoul_gu(df, job_name)
    
    # GeoJSON 이름 컬럼 찾기
    name_col = None
    for col in ['name', 'SIG_KOR_NM', 'GU_NM', 'kor_name', 'NAME_2', 'sggnm']:
        if col in seoul_gdf_raw.columns:
            name_col = col
            break
    if name_col is None:
        print("⚠ 구 이름 컬럼을 찾지 못했습니다. 컬럼 목록:", seoul_gdf_raw.columns.tolist())
        return
    
    seoul_gdf = seoul_gdf_raw.copy()
    seoul_gdf['gu_std'] = seoul_gdf[name_col].str.strip()
    
    # 병합
    seoul_gdf = seoul_gdf.merge(agg, left_on='gu_std', right_on='gu', how='left')
    seoul_gdf['count'] = seoul_gdf['count'].fillna(0)
    seoul_gdf['pct']   = seoul_gdf['pct'].fillna(0)
    
    # ── 그리기 ──
    fig, ax = plt.subplots(1, 1, figsize=(14, 12), facecolor='#FAFAFA')
    ax.set_facecolor('#FAFAFA')
    
    vmax = seoul_gdf['count'].max()
    vmin = 0
    
    seoul_gdf.plot(
        column='count',
        cmap=cfg['cmap'],
        linewidth=0.8,
        edgecolor='#888888',
        ax=ax,
        vmin=vmin,
        vmax=vmax,
        missing_kwds={'color': '#EEEEEE'},
    )
    
    # 레이블
    for _, row in seoul_gdf.iterrows():
        centroid = row.geometry.centroid
        x, y = centroid.x, centroid.y
        
        norm_val = row['count'] / vmax if vmax > 0 else 0
        text_color = 'white' if norm_val > 0.40 else '#1A1A1A'
        
        gu_name = row['gu_std']
        if row['count'] > 0:
            label = f"{gu_name}\n{int(row['count'])}건\n({row['pct']}%)"
        else:
            label = f"{gu_name}\n0건"
        
        ax.annotate(
            label,
            xy=(x, y),
            ha='center', va='center',
            fontsize=7.5,
            color=text_color,
            fontweight='bold',
        )
    
    # 컬러바
    sm = plt.cm.ScalarMappable(
        cmap=cfg['cmap'],
        norm=mcolors.Normalize(vmin=vmin, vmax=vmax)
    )
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02, shrink=0.6)
    cbar.set_label('공고 건수', fontsize=11)
    cbar.ax.tick_params(labelsize=9)
    
    # 강남구 강조 표시 (발표 포인트)
    gangnam = seoul_gdf[seoul_gdf['gu_std'] == '강남구']
    if not gangnam.empty:
        gangnam.boundary.plot(ax=ax, linewidth=2.5, edgecolor='#FF4444')
    
    title_map = {'데이터 분석가': '데이터 분석가', '백엔드 개발자': '백엔드 개발자'}
    ax.set_title(
        f'[서울 구별] {title_map[job_name]} 채용공고 분포\n(서울 내 총 {total_seoul}건)',
        fontsize=16, fontweight='bold', pad=18, color='#1A1A1A'
    )
    ax.axis('off')
    
    plt.tight_layout()
    fname = f"{cfg['filename_prefix']}_seoul_map.png"
    fpath = os.path.join(output_dir, fname)
    plt.savefig(fpath, dpi=200, bbox_inches='tight', facecolor='#FAFAFA')
    plt.close()
    print(f"저장: {fpath}")


# ────────────────────────────────────────────────
# 7. 실행
# ────────────────────────────────────────────────
def load_geojson_or_shp(path):
    """GeoJSON / SHP 자동 로드"""
    gdf = gpd.read_file(path)
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    elif gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs("EPSG:4326")
    return gdf

def main():
    # ── 지도 데이터 로드 ──
    if not os.path.exists(SIDO_SHP):
        print(f"[ERROR] 시도 파일이 없습니다: {SIDO_SHP}")
        print("아래 명령어로 다운로드하세요:")
        print("  curl -L -o sido.geojson https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2018/json/skorea-provinces-2018-geo.json")
        return
    if not os.path.exists(SEOUL_GU_SHP):
        print(f"[ERROR] 서울 구 파일이 없습니다: {SEOUL_GU_SHP}")
        print("아래 명령어로 다운로드하세요:")
        print("  curl -L -o seoul_gu.geojson https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2018/json/skorea-municipalities-2018-geo.json")
        print("  (서울 구만 필터 필요시 코드 내 seoul_gu_gdf = gdf[gdf['name'].str.contains('서울')] 추가)")
        return
    
    print("시도 지도 로드 중...")
    sido_gdf = load_geojson_or_shp(SIDO_SHP)
    print(f"  - 시도 수: {len(sido_gdf)}, 컬럼: {sido_gdf.columns.tolist()}")
    
    print("서울 구 지도 로드 중...")
    gu_gdf_all = load_geojson_or_shp(SEOUL_GU_SHP)
    print(f"  - 전체 시군구 수: {len(gu_gdf_all)}, 컬럼: {gu_gdf_all.columns.tolist()}")
    
    # 서울 구만 필터링
    name_col_gu = None
    for col in ['name', 'SIG_KOR_NM', 'GU_NM', 'sggnm', 'NAME_2']:
        if col in gu_gdf_all.columns:
            name_col_gu = col
            break
    
    if name_col_gu:
        # 서울특별시 소속 구만
        if 'CTPRVN_CD' in gu_gdf_all.columns:
            seoul_gu_gdf = gu_gdf_all[gu_gdf_all['CTPRVN_CD'] == '11'].copy()
        elif 'sido' in gu_gdf_all.columns:
            seoul_gu_gdf = gu_gdf_all[gu_gdf_all['sido'].str.contains('서울', na=False)].copy()
        elif 'code' in gu_gdf_all.columns:
            seoul_gu_gdf = gu_gdf_all[gu_gdf_all['code'].astype(str).str.startswith('11')].copy()
        else:
            # 구 이름으로 필터 (서울 25개 구)
            SEOUL_GU_LIST = [
                '강남구','강동구','강북구','강서구','관악구','광진구','구로구','금천구',
                '노원구','도봉구','동대문구','동작구','마포구','서대문구','서초구',
                '성동구','성북구','송파구','양천구','영등포구','용산구','은평구',
                '종로구','중구','중랑구'
            ]
            seoul_gu_gdf = gu_gdf_all[gu_gdf_all[name_col_gu].isin(SEOUL_GU_LIST)].copy()
        print(f"  - 서울 구 수: {len(seoul_gu_gdf)}")
    else:
        seoul_gu_gdf = gu_gdf_all
        print("⚠ 구 이름 컬럼을 찾지 못해 전체 데이터를 사용합니다.")
    
    # ── 지도 생성 ──
    for job_name in ['데이터 분석가', '백엔드 개발자']:
        print(f"\n===== {job_name} 지도 생성 =====")
        draw_sido_map(job_name, sido_gdf, df, OUTPUT_DIR)
        draw_seoul_map(job_name, seoul_gu_gdf, df, OUTPUT_DIR)
    
    print(f"\n✅ 완료! 저장 위치: {os.path.abspath(OUTPUT_DIR)}/")
    print("   파일 목록:")
    for f in sorted(os.listdir(OUTPUT_DIR)):
        print(f"   - {f}")

if __name__ == "__main__":
    main()


국내 데이터: 877건
job
백엔드 개발자    603
데이터 분석가    274
Name: count, dtype: int64
폰트 설정: C:/Windows/Fonts/malgun.ttf
시도 지도 로드 중...
  - 시도 수: 17, 컬럼: ['name', 'base_year', 'name_eng', 'code', 'geometry']
서울 구 지도 로드 중...
  - 전체 시군구 수: 250, 컬럼: ['name', 'base_year', 'name_eng', 'code', 'geometry']
  - 서울 구 수: 25

===== 데이터 분석가 지도 생성 =====
저장: C:\py_temp\중간프로젝트\output_maps_m1\DA_sido_map.png
저장: C:\py_temp\중간프로젝트\output_maps_m1\DA_seoul_map.png

===== 백엔드 개발자 지도 생성 =====
저장: C:\py_temp\중간프로젝트\output_maps_m1\BE_sido_map.png
저장: C:\py_temp\중간프로젝트\output_maps_m1\BE_seoul_map.png

✅ 완료! 저장 위치: C:\py_temp\중간프로젝트\output_maps_m1/
   파일 목록:
   - BE_seoul_map.png
   - BE_sido_map.png
   - DA_seoul_map.png
   - DA_sido_map.png
